# 07 — Evaluator demo and final results export

**Viraat Chaudhary · DATA 266 Lab 1 · Team 15 · Yelp Polarity**

Complete notebook 06 including your manual annotations and findings. CPU is sufficient. This demonstrates saved checkpoints on synthetic sentences; it does not measure new test accuracy. Have your downloaded executed notebooks 02–06 ready to upload in the collection cell. Notebook 01 is already preserved.

Run top-to-bottom. Save the executed notebook using **File → Download → Download .ipynb**. Use the exact filename provided. Outputs persist in your own Drive run folder. Source code is assistant-assisted; student explanations and analysis must be your own.


In [1]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from google.colab import drive, files

# Set these once; keep the same RUN_NAME for notebooks 02–07 and resumed sessions.
DRIVE_BASE_RELATIVE = 'DATA266_Lab1_Task2/Viraat_Chaudhary'
RUN_NAME = 'task2_models_rnn_lstm_tcn_v1'
PACKAGE_ZIP_NAME = 'Viraat_Task2_Run_Package.zip'
PACKAGE_ID = 'viraat_yelp_rnn_lstm_tcn_v1'

drive.mount('/content/drive', force_remount=False)
DRIVE_BASE = Path('/content/drive/MyDrive') / DRIVE_BASE_RELATIVE
ROOT = DRIVE_BASE / RUN_NAME / 'task2_sentiment/Viraat_Chaudhary'
CACHE_ROOT = Path('/content/viraat_task2_data')
if not ROOT.exists():
    archive = DRIVE_BASE / PACKAGE_ZIP_NAME
    if not archive.exists():
        print('Upload Viraat_Task2_Run_Package.zip (the code package, not a dataset ZIP).')
        uploaded = files.upload()
        if PACKAGE_ZIP_NAME not in uploaded:
            raise FileNotFoundError('Select the exact package ZIP filename.')
        archive = Path('/content') / PACKAGE_ZIP_NAME
        archive.write_bytes(uploaded[PACKAGE_ZIP_NAME])
    extraction = Path('/content/viraat_task2_package_extract')
    extraction.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for info in bundle.infolist():
            if not (extraction / info.filename).resolve().is_relative_to(extraction.resolve()):
                raise ValueError('Invalid package ZIP path.')
        if bundle.testzip() is not None:
            raise ValueError('Package ZIP CRC failed.')
        bundle.extractall(extraction)
    manifest = json.loads((extraction / 'PACKAGE_CONTENT_MANIFEST.json').read_text())
    if manifest['package_id'] != PACKAGE_ID:
        raise ValueError('Incorrect package identity.')
    for item in manifest['files']:
        path = extraction / item['path']
        if hashlib.sha256(path.read_bytes()).hexdigest() != item['sha256']:
            raise ValueError('Package integrity failed: ' + item['path'])
    source = extraction / 'task2_sentiment/Viraat_Chaudhary'
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, ROOT)
    (ROOT / 'package_installation.json').write_text(json.dumps({'package_id': PACKAGE_ID,
                                                              'package_manifest_sha256': hashlib.sha256(
        (extraction / 'PACKAGE_CONTENT_MANIFEST.json').read_bytes()).hexdigest()}, indent=2) + '\n')
else:
    installed = json.loads((ROOT / 'package_installation.json').read_text())
    if installed['package_id'] != PACKAGE_ID:
        raise ValueError('This RUN_NAME belongs to a different package.')

required = {'numpy': 'numpy==2.1.3', 'scipy': 'scipy>=1.14,<2',
            'sklearn': 'scikit-learn==1.6.1', 'nltk': 'nltk==3.9.1',
            'datasets': 'datasets==4.8.5', 'matplotlib': 'matplotlib==3.10.0',
            'pandas': 'pandas>=2.2,<4', 'tqdm': 'tqdm==4.67.3'}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Use a Colab runtime with PyTorch installed.')
# Keep the tokenizer version consistent with the approved snapshot.
import importlib.metadata
if importlib.metadata.version('nltk') != '3.9.1':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nltk==3.9.1'])

source_dir = str(ROOT / 'src')
sys.path.insert(0, source_dir)
for name in ('runtime_utils', 'yelp_data', 'data_runtime', 'models', 'metrics',
             'training', 'evaluation', 'review', 'reporting', 'evaluator'):
    module = sys.modules.get(name)
    if module is not None and Path(getattr(module, '__file__', '')).parent != ROOT / 'src':
        del sys.modules[name]
from data_runtime import verify_preprocessing
verify_preprocessing(ROOT)
import torch
import pandas as pd
print('Member folder:', ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Approved preprocessing restored; original notebook 01 is preserved.')


Mounted at /content/drive
Member folder: /content/drive/MyDrive/DATA266_Lab1_Task2/Viraat_Chaudhary/task2_models_rnn_lstm_tcn_v1/task2_sentiment/Viraat_Chaudhary
GPU: NVIDIA A100-SXM4-40GB
Approved preprocessing restored; original notebook 01 is preserved.


In [2]:
from evaluator import smoke_all
SMOKE = smoke_all(ROOT)
print(SMOKE['status'])
for name, rows in SMOKE['predictions'].items():
    print(name)
    display(pd.DataFrame(rows))


PASS
baseline


,text,probability_positive,label
0,The food was delicious and the service was fri...,0.999848,positive
1,The food was not good and the service was terr...,0.000142,negative
2,Great food.\nClean café.,0.992270,positive
3,,0.160104,negative


experiment_1


,text,probability_positive,label
0,The food was delicious and the service was fri...,0.998021,positive
1,The food was not good and the service was terr...,0.000005,negative
2,Great food.\nClean café.,0.990206,positive
3,,0.038641,negative


experiment_2


,text,probability_positive,label
0,The food was delicious and the service was fri...,0.999082,positive
1,The food was not good and the service was terr...,0.000081,negative
2,Great food.\nClean café.,0.999350,positive
3,,0.105465,negative


## Collect executed notebooks 02–06

Upload the five executed notebook downloads, with their outputs saved. Copies with a download suffix such as (1) are accepted and renamed to the agreed names. The verified notebook 01 is not replaced. Download the executed notebook 07 after its final cell; send it separately with the final results ZIP, since a notebook cannot include its own completed future output.

In [3]:
print('Upload the five executed notebooks 02–06 together.')
notebooks = files.upload()
expected = ['Viraat_02_train_baseline.ipynb','Viraat_03_train_experiment_1.ipynb','Viraat_04_train_experiment_2.ipynb','Viraat_05_evaluate_compare_models.ipynb','Viraat_06_manual_error_analysis.ipynb']
received = set()
for filename, contents in notebooks.items():
    match = next((n for n in expected if filename.startswith(Path(n).stem) and filename.endswith('.ipynb')), None)
    if match is None:
        raise ValueError('Unexpected notebook: ' + filename)
    notebook = json.loads(contents.decode('utf-8'))
    cells = [c for c in notebook['cells'] if c['cell_type'] == 'code']
    if any(c.get('execution_count') is None for c in cells):
        raise ValueError('This notebook contains unexecuted code cells: ' + filename)
    if any(o.get('output_type') == 'error' for c in cells for o in c.get('outputs', [])):
        raise ValueError('This notebook still contains a saved exception: ' + filename)
    stage = int(match.split('_')[1])
    if notebook.get('metadata', {}).get('task2_stage') != stage:
        raise ValueError('Notebook stage does not match filename: ' + filename)
    (ROOT / 'notebooks' / match).write_bytes(contents)
    received.add(match)
if received != set(expected):
    raise ValueError('Upload all five executed notebooks 02–06.')
print('Executed notebook copies saved.')


Upload the five executed notebooks 02–06 together.


Saving Viraat_02_train_baseline.ipynb to Viraat_02_train_baseline.ipynb
Saving Viraat_03_train_experiment_1.ipynb to Viraat_03_train_experiment_1.ipynb
Saving Viraat_04_train_experiment_2.ipynb to Viraat_04_train_experiment_2.ipynb
Saving Viraat_05_evaluate_compare_models.ipynb to Viraat_05_evaluate_compare_models.ipynb
Saving Viraat_06_manual_error_analysis.ipynb to Viraat_06_manual_error_analysis.ipynb
Executed notebook copies saved.


## Audit and export all final results

The ZIP includes three selected checkpoints, source/configs, logs, full metrics, predictions, plots, twenty reviewed errors, student findings and executed notebooks 01–06. Datasets and large optimizer resume checkpoints are excluded. Add your separately downloaded executed 07 when preparing the final repository. Shared README additions and combined PDF report still require team integration.

In [4]:
from reporting import refresh_reports, submission_audit, export_artifacts
refresh_reports(ROOT)
AUDIT = submission_audit(ROOT)
print(json.dumps(AUDIT, indent=2))
if not AUDIT['task2_individual_ready_for_review']:
    raise RuntimeError('Complete the missing individual artifacts shown in the audit before export.')
FINAL_ZIP = export_artifacts(ROOT, 'Run_Results', include_checkpoints=True)
print('Final results:', FINAL_ZIP)
files.download(str(FINAL_ZIP))
print('Now download executed Viraat_07_evaluator_demo.ipynb and send it with this ZIP for review.')


{
  "task2_individual_ready_for_review": true,
  "individual_submission_files_complete": false,
  "full_lab_submission_ready": false,
  "missing_required_files": [],
  "student_analysis_pending": false,
  "executed_notebook_copies_still_needed": [
    "Viraat_07_evaluator_demo.ipynb"
  ],
  "notebooks_with_saved_errors": [],
  "checkpoints_over_github_limit": [],
  "remaining_team_work": [
    "confirm teammate exact hardware/provenance",
    "jointly write task comparison and integrate combined lab PDF report"
  ],
  "datasets_in_submission_bundle": false,
  "created_utc": "2026-10-02T02:15:44.373240+00:00"
}
Final results: /content/drive/MyDrive/DATA266_Lab1_Task2/Viraat_Chaudhary/task2_models_rnn_lstm_tcn_v1/task2_sentiment/Viraat_Chaudhary/exports/Viraat_Task2_Run_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Now download executed Viraat_07_evaluator_demo.ipynb and send it with this ZIP for review.
